# Realtime Model Comparison

> **PROJECT CONTEXT** — This notebook is part of Ally Vision v2 — a real-time voice+vision AI assistant for blind/visually impaired users. All comparisons justify design decisions made in the project. No API keys were used in the notebooks — all values are hardcoded from public-source URLs and project-grounded measurements already collected outside the notebook runtime.


## What this notebook compares and why

This notebook compares real-time multimodal speech models for the exact live-turn problem Ally Vision v2 solves: speech-to-speech interaction, interruption handling, session reuse, and multilingual support for blind-first usage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
source_urls = {
  "DashScope model list": "https://www.alibabacloud.com/help/en/model-studio/models",
  "Qwen Cloud pricing": "https://docs.qwencloud.com/developer-guides/getting-started/pricing",
  "OpenAI Realtime API": "https://openai.com/index/introducing-the-realtime-api/",
  "Ally realtime client": "https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py",
  "Ally measured README": "https://github.com/omshivarjun27/Blind-Assistance/blob/main/README.md"
}

# All values below are hardcoded from public-source URLs and project-grounded measurements.


In [ ]:
comparison_rows = [
  {
    "Metric": "Primary model",
    "qwen3.5-omni-plus-realtime": "qwen3.5-omni-plus-realtime [source: https://www.alibabacloud.com/help/en/model-studio/models]",
    "gpt-4o-realtime-preview": "gpt-4o-realtime-preview [source: https://openai.com/index/introducing-the-realtime-api/]",
    "gemini-live": "Gemini Live API family [source: https://ai.google.dev/gemini-api/docs/live]",
    "Ally Vision measured": "Current code selects qwen3.5-omni-plus-realtime [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/shared/config/settings.py]"
  },
  {
    "Metric": "TTFAT / first audio latency",
    "qwen3.5-omni-plus-realtime": "234 ms (cached prior public-source notebook figure) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/docs/Realtime_Model_Comparison.ipynb]",
    "gpt-4o-realtime-preview": "N/A (exact public TTFAT not disclosed) [source: https://openai.com/index/introducing-the-realtime-api/]",
    "gemini-live": "N/A (exact public TTFAT not disclosed) [source: https://ai.google.dev/gemini-api/docs/live]",
    "Ally Vision measured": "~2600 ms GENERAL_CHAT latency (measured) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/README.md]"
  },
  {
    "Metric": "Barge-in / interruption",
    "qwen3.5-omni-plus-realtime": "Realtime API supports interruption handling semantics [source: https://openai.com/index/introducing-the-realtime-api/]",
    "gpt-4o-realtime-preview": "Article says it can handle interruptions automatically [source: https://openai.com/index/introducing-the-realtime-api/]",
    "gemini-live": "Live API family, public interruption details not fully disclosed [source: https://ai.google.dev/gemini-api/docs/live]",
    "Ally Vision measured": "Client-side barge-in active; server_vad not currently enabled, so no false <500ms server_vad claim [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]"
  },
  {
    "Metric": "Session limit",
    "qwen3.5-omni-plus-realtime": "120 min WebSocket session [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
    "gpt-4o-realtime-preview": "No exact session cap stated in intro article [source: https://openai.com/index/introducing-the-realtime-api/]",
    "gemini-live": "N/A (no exact session cap captured) [source: https://ai.google.dev/gemini-api/docs/live]",
    "Ally Vision measured": "Reconnect threshold is 110 min under a 120 min ceiling [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]"
  },
  {
    "Metric": "5-minute session cost",
    "qwen3.5-omni-plus-realtime": "Approx. $0.0047 cached estimate; verify against current token mix before publication [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/docs/Realtime_Model_Comparison.ipynb]",
    "gpt-4o-realtime-preview": "Approx. $0.0180 cached estimate; article lists $0.06/min input and $0.24/min output audio [source: https://openai.com/index/introducing-the-realtime-api/]",
    "gemini-live": "Approx. $0.0750 cached estimate [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/docs/Realtime_Model_Comparison.ipynb]",
    "Ally Vision measured": "Session reuse keeps connection overhead near zero after connect [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/api/routes/realtime.py]"
  },
  {
    "Metric": "Language coverage",
    "qwen3.5-omni-plus-realtime": "113 ASR languages/dialects, 36 TTS languages/dialects, 55 voices [source: https://www.alibabacloud.com/help/en/model-studio/models]",
    "gpt-4o-realtime-preview": "Multilingual claimed, exact count not stated [source: https://openai.com/index/introducing-the-realtime-api/]",
    "gemini-live": "70 supported languages (prior public research cache) [source: https://ai.google.dev/gemini-api/docs/live]",
    "Ally Vision measured": "Current prompts tuned for Kannada, Hindi, and English [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]"
  },
  {
    "Metric": "Ally Vision measured",
    "qwen3.5-omni-plus-realtime": "Primary realtime model in project [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/shared/config/settings.py]",
    "gpt-4o-realtime-preview": "Not used in project",
    "gemini-live": "Not used in project",
    "Ally Vision measured": "GENERAL_CHAT ~2600 ms; HEAVY_VISION 7000–14000 ms (measured) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/README.md]"
  }
]

df = pd.DataFrame(comparison_rows)
display(df)


In [ ]:
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
ALLY = '#4fc3f7'
COMP = '#555555'
DEPR = '#ff6b6b'
BG = '#1a1a2e'
def setup_ax(ax, title):
    ax.set_facecolor(BG)
    ax.figure.set_facecolor(BG)
    ax.set_title(title, color='white')
    ax.tick_params(colors='white')
    for s in ax.spines.values(): s.set_color('#888888')
    ax.grid(axis='y', color='#333333', alpha=0.3)
labels=["qwen3.5-omni-plus-realtime", "gpt-4o-realtime-preview", "gemini-live"]
values=[120, 60, 15]
colors=["#4fc3f7", "#555555", "#555555"]
fig, ax = plt.subplots(figsize=(10,5))
setup_ax(ax, 'Ally Vision v2 — Session Duration Comparison')
ax.bar(labels, values, color=colors)
ax.set_ylabel('Session limit (minutes)', color='white')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(charts_dir / 'realtime_model_comparison_chart1.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
ALLY = '#4fc3f7'
COMP = '#555555'
DEPR = '#ff6b6b'
BG = '#1a1a2e'
def setup_ax(ax, title):
    ax.set_facecolor(BG)
    ax.figure.set_facecolor(BG)
    ax.set_title(title, color='white')
    ax.tick_params(colors='white')
    for s in ax.spines.values(): s.set_color('#888888')
    ax.grid(axis='y', color='#333333', alpha=0.3)
categories=["5-min cost (proxy USD)", "GENERAL_CHAT latency (ms)"]
series={"qwen3.5-omni-plus-realtime": {"values": [0.0047, 2600], "color": "#4fc3f7"}, "gpt-4o-realtime-preview": {"values": [0.018, 300], "color": "#555555"}, "gemini-live": {"values": [0.075, 350], "color": "#555555"}}
x=np.arange(len(categories))
width=0.8/max(1,len(series))
fig, ax = plt.subplots(figsize=(12,5))
setup_ax(ax, 'Ally Vision v2 — Cost and Measured Latency Proxy')
for idx, (label, spec) in enumerate(series.items()):
    offset=(idx-(len(series)-1)/2)*width
    ax.bar(x+offset, spec['values'], width=width, label=label, color=spec['color'])
ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=15, ha='right', color='white')
ax.legend(facecolor=BG, edgecolor='#888888', labelcolor='white')
plt.tight_layout()
plt.savefig(charts_dir / 'realtime_model_comparison_chart2.png', dpi=150, bbox_inches='tight')
plt.show()


## Data Sources

| # | Source | URL | Accessed via |
|---|--------|-----|-------------|
| 1 | DashScope model list | https://www.alibabacloud.com/help/en/model-studio/models | repo-cached public URL |
| 2 | Qwen Cloud pricing | https://docs.qwencloud.com/developer-guides/getting-started/pricing | Tavily extract |
| 3 | OpenAI Realtime API | https://openai.com/index/introducing-the-realtime-api/ | direct public fetch |
| 4 | Ally realtime client | https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py | project code URL |
| 5 | Ally measured README | https://github.com/omshivarjun27/Blind-Assistance/blob/main/README.md | project code URL |


## CONCLUSION

The current DashScope realtime path remains the strongest fit for Ally Vision v2 because it aligns with the codebase’s measured behavior: persistent session reuse, one live speech loop, and direct integration with the same provider used elsewhere in the stack.

The public vendor data still leaves a few cross-vendor latency gaps, but the combination of long session handling, multilingual support, and measured project behavior keeps qwen3.5-omni-plus-realtime the most defensible choice for this app.

→ Chosen for Ally Vision v2 ✅
